In [1]:
import requests
import json
from bs4 import BeautifulSoup
import pandas as pd
# import warnings
URL = "https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doPmr=yes"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/151.0.0.0 Safari/537.36"
    )
}

from utils import Helper

utils = Helper()
managers = utils.load_json(r"manager.json")

In [2]:
session = requests.Session()

resp = session.get(URL, headers=headers, verify=False)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")

token_tag = soup.find("input", {"name": "org.apache.struts.taglib.html.TOKEN"})
if not token_tag:
    raise Exception("Token not found")

token = token_tag["value"]
payload = {
    "currdate": "",
    "loginflag": "0",
    "searchValue": "",
    "pmrId": "INP000008464@@INP000008464@@1729 ADVISORS LLP",
    "year": "2026",
    "month": "2",
    "format": "",
    "org.apache.struts.taglib.html.TOKEN": token,
    "loginEmail": "",
    "loginPassword": "",
    "cap_login": "",
    "moduleNo": "-1",
    "moduleId": "",
    "link": "",
    "yourName": "",
    "friendName": "",
    "friendEmail": "",
    "mailmessage": "",
    "cap_email": ""
}
post_headers = {
    **headers,
    "Referer": URL,
    "Origin": "https://www.sebi.gov.in"
}

year = {
    "2026": "2026",
    # "2025": "2025",
    # "2024": "2024",
    # "2023": "2023",
    # "2022": "2022",
    # "2021": "2021",
    # "2020": "2020",
    # "2019": "2019",
    # "2018": "2018"
}
month =  {
    # "1": "January",
    # "2": "February",
    # "3": "March",
    # "4": "April",
    # "5": "May",
    # "6": "June",
    # "7": "July",
    "8": "August",
    # "9": "September",
    # "10": "October",
    # "11": "November",
    # "12": "December"
}

manager = utils.load_json(r"manager.json")

c:\Users\kaustubh.keny\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.sebi.gov.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [7]:
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO
import warnings

warnings.filterwarnings("ignore")

all_dfs = []

for id, manager in enumerate(managers):

    company_name = manager["company_name"]
    pmr_id = manager["raw"]
    
    print(f"================== fetch: {manager} =================")

    for yr in year:
        for mon in month:
            
            print(f"{yr} -- {mon}")
            
            if id%10 == 0:
            
                search_resp = session.post(URL, headers=post_headers,verify=False)
                search_resp.raise_for_status()
                
                soup = BeautifulSoup(resp.text, "html.parser")

                token_tag = soup.find("input", {"name": "org.apache.struts.taglib.html.TOKEN"})
                if not token_tag:
                    raise Exception("Token not found")

                token = token_tag["value"]
            
            payload = {
                "currdate": "",
                "loginflag": "0",
                "searchValue": "",
                "pmrId": pmr_id,
                "year": yr,
                "month": mon,
                "format": "",
                "org.apache.struts.taglib.html.TOKEN": token,
                "loginEmail": "",
                "loginPassword": "",
                "cap_login": "",
                "moduleNo": "-1",
                "moduleId": "",
                "link": "",
                "yourName": "",
                "friendName": "",
                "friendEmail": "",
                "mailmessage": "",
                "cap_email": ""
            }

            resp = session.post(
                URL,
                headers=post_headers,
                data=payload,
                verify=False
            )

            resp.raise_for_status()

            try:
                with open(f"sebi_{company_name}_{yr}_{mon}.html", "w", encoding="utf-8") as f:
                    f.write(resp.text)
                tables = pd.read_html(StringIO(resp.text), header=None)

                for idx, df in enumerate(tables):
                    df = df.copy()

                    # Convert all values to strings, treat as raw datapoints
                    df = df.astype(str)

                    # Reset columns to simple integer positions
                    df.columns = range(df.shape[1])

                    # Add metadata columns
                    df.insert(0, "company_name", company_name)
                    df.insert(1, "year", yr)
                    df.insert(2, "month", month[mon])
                    df.insert(3, "table_no", idx + 1)

                    all_dfs.append(df)

            except ValueError:
                # No tables found, skip
                pass

================== fetch: {'company_name': '1729 ADVISORS LLP', 'raw': 'INP000008464@@INP000008464@@1729 ADVISORS LLP'} =================
2026 -- 8
================== fetch: {'company_name': '2POINT2 CAPITAL ADVISORS LLP', 'raw': 'INP000005190@@INP000005190@@2POINT2 CAPITAL ADVISORS LLP'} =================
2026 -- 8
================== fetch: {'company_name': '360 ONE ALTERNATES ASSET MANAGEMENT LIMITED', 'raw': 'INP000008640@@INP000008640@@360 ONE ALTERNATES ASSET MANAGEMENT LIMITED'} =================
2026 -- 8


KeyboardInterrupt: 

In [6]:
# Final dataframe
final_df = pd.concat(all_dfs, ignore_index=True, sort=False)
final_df.to_excel("sebi_output_2.xlsx", index=False)

In [ ]:
import pandas as pd
from collections import defaultdict
from bs4 import BeautifulSoup
import requests
import json
from bs4 import BeautifulSoup
import pandas as pd
# import warnings
URL = "https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doPmr=yes"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/151.0.0.0 Safari/537.36"
    )
}

from utils import Helper

utils = Helper()
managers = utils.load_json(r"manager.json")

session = requests.Session()

resp = session.get(URL, headers=headers, verify=False)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")

token_tag = soup.find("input", {"name": "org.apache.struts.taglib.html.TOKEN"})
if not token_tag:
    raise Exception("Token not found")

token = token_tag["value"]

payload = {
    "currdate": "",
    "loginflag": "0",
    "searchValue": "",
    "pmrId": "INP000008464@@INP000008464@@1729 ADVISORS LLP",
    "year": "2026",
    "month": "2",
    "format": "",
    "org.apache.struts.taglib.html.TOKEN": token,
    "loginEmail": "",
    "loginPassword": "",
    "cap_login": "",
    "moduleNo": "-1",
    "moduleId": "",
    "link": "",
    "yourName": "",
    "friendName": "",
    "friendEmail": "",
    "mailmessage": "",
    "cap_email": ""
}

post_headers = {
    **headers,
    "Referer": URL,
    "Origin": "https://www.sebi.gov.in"
}



# =====================================================
# Table-wise storage
# =====================================================

sheet_data = defaultdict(list)

year = {
    "2026": "2026",
    # "2025": "2025",
    # "2024": "2024",
    # "2023": "2023",
    # "2022": "2022",
    # "2021": "2021",
    # "2020": "2020",
    # "2019": "2019",
    # "2018": "2018"
}
month =  {
    "1": "January",
    # "2": "February",
    # "3": "March",
    # "4": "April",
    # "5": "May",
    # "6": "June",
    # "7": "July",
    # "8": "August",
    # "9": "September",
    # "10": "October",
    # "11": "November",
    # "12": "December"
}

for manager_idx, manager in enumerate(managers):



    if manager_idx <350:
        continue
    
    company_name = manager["company_name"]
    pmr_id = manager["raw"]

    print(f"Processing: {company_name}")

    for yr in year:
        for mon in month:

            # Refresh token periodically
            if manager_idx % 10 == 0:

                search_resp = session.post(URL, headers=post_headers, verify=False)
                search_resp.raise_for_status()

                soup = BeautifulSoup(search_resp.text, "html.parser")

                token_tag = soup.find(
                    "input", {"name": "org.apache.struts.taglib.html.TOKEN"}
                )

                if not token_tag:
                    raise Exception("Token not found")

                token = token_tag["value"]

            payload = {
                "currdate": "",
                "loginflag": "0",
                "searchValue": "",
                "pmrId": pmr_id,
                "year": yr,
                "month": mon,
                "format": "",
                "org.apache.struts.taglib.html.TOKEN": token,
                "loginEmail": "",
                "loginPassword": "",
                "cap_login": "",
                "moduleNo": "-1",
                "moduleId": "",
                "link": "",
                "yourName": "",
                "friendName": "",
                "friendEmail": "",
                "mailmessage": "",
                "cap_email": "",
            }

            resp = session.post(URL, headers=post_headers, data=payload, verify=False)

            resp.raise_for_status()

            try:

                tables = pd.read_html(resp.text, header=None)

            except ValueError:
                continue

            # ==========================================
            # Process each table
            # ==========================================

            for table_no, df in enumerate(tables, start=1):

                df = df.copy()

                # Flatten multi-level headers
                if isinstance(df.columns, pd.MultiIndex):

                    header_vals = [
                        " | ".join(str(x) for x in col if pd.notna(x))
                        for col in df.columns
                    ]

                else:

                    header_vals = [str(x) for x in df.columns]

                # Move header row into data
                df.columns = range(df.shape[1])

                header_row = pd.DataFrame([header_vals], columns=df.columns)

                df = pd.concat([header_row, df], ignore_index=True)

                # Metadata
                df.insert(0, "manager_name", company_name)
                df.insert(1, "year", yr)
                df.insert(2, "month", month[mon])

                sheet_data[f"Table_{table_no}"].append(df)


# =====================================================
# Write Excel
# =====================================================

output_excel = "manager_tables.xlsx"

with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:

    for s_name in sorted(sheet_data.keys(), key=lambda x: int(x.split("_")[1])):

        final_df = pd.concat(sheet_data[s_name], ignore_index=True)
        final_df.to_excel(writer, sheet_name=s_name[:31], index=False)

        print(f"{s_name}: " f"{len(final_df):,} rows")

print(f"\nSaved: {output_excel}")


In [8]:
output_excel = "manager_tables_2.xlsx"

with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:

    for s_name in sorted(sheet_data.keys(), key=lambda x: int(x.split("_")[1])):
        final_df = pd.concat(sheet_data[s_name], ignore_index=True)
        final_df.to_excel(writer, sheet_name=s_name[:31], index=False)

        print(f"{s_name}: " f"{len(final_df):,} rows")

print(f"\nSaved: {output_excel}")

Table_1: 229 rows
Table_2: 687 rows
Table_3: 1,145 rows
Table_4: 1,145 rows
Table_5: 687 rows
Table_6: 1,202 rows
Table_7: 1,432 rows
Table_8: 916 rows
Table_9: 2,028 rows
Table_10: 972 rows
Table_11: 687 rows
Table_12: 458 rows
Table_13: 687 rows
Table_14: 916 rows
Table_15: 458 rows
Table_16: 687 rows
Table_17: 687 rows
Table_18: 458 rows
Table_19: 2,061 rows

Saved: manager_tables_2.xlsx
